# Qwen3 Position-Geometry Localization
## v0.6.2 — query displacement vs irrelevant-filler positional spreading

v0.6.1 prospectively replicated the revised positional finding on new seeds:

- expanding position geometry improved the gold-vs-best-distractor margin with canonical token IDs;
- it improved the margin even more strongly with fragmented filler IDs;
- both effects were positive in all four language×length cells and leave-one-trial-out sign-stable.

That result is retained as a successful **prospective replication of a positional effect**, not as support for the original degradation→rescue hypothesis.

v0.6.2 asks the next mechanistic question:

> **Which part of the expanded geometry produces the benefit?**

The v0.6.1 expansion simultaneously did two things:

1. it moved the query/readout farther from the evidence;
2. it spread the irrelevant filler tokens across the enlarged positional span.

This notebook separates those components using six conditions and **32 entirely new semantic seeds**.

## Six-condition geometry design

For canonical token IDs:

- `CC`: canonical/source geometry;
- `CQ`: keep evidence+filler positions canonical, but shift query/readout to the expanded absolute positions;
- `CE`: keep the same shifted query/readout as `CQ`, and additionally spread filler positions across the expanded span.

For fragmented filler IDs:

- `FC`: source-aligned geometry;
- `FQ`: keep source-aligned filler positions, but shift query/readout to the expanded absolute positions;
- `FE`: ordinary expanded geometry.

Critical same-ID contrasts:

- `CQ - CC`: query/readout displacement effect with canonical IDs;
- `CE - CQ`: incremental filler-spreading effect with canonical IDs;
- `FQ - FC`: query/readout displacement effect with fragmented IDs;
- `FE - FQ`: incremental filler-spreading effect with fragmented IDs.

Importantly, `CQ` and `CE` have:

- identical canonical token IDs;
- identical query/readout positions;
- identical global evidence→readout distance;

and differ **only in the internal positions of the irrelevant filler**.

The same is true for `FQ` vs `FE`.

## Primary metric

Gold-vs-best-distractor candidate logit margin.

Higher is better.

Forced-choice NLL and accuracy remain secondary diagnostics.

## Prospective status

Seeds `3000..3031` are new and were not used in v0.5, v0.6.0, or v0.6.1.

The purpose is localization, not post-hoc rescue of any earlier hypothesis.

> Public reproduction copy: calibration values are frozen explicitly; calibration scans and development/debugging history are not required to reproduce the paper's reported experiment.

## 0. T4 runtime

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 1. Drive, provenance, and frozen settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, shutil

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3_position_localization_v062')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = PROJECT_DIR / 'v062_geometry_localization_4k8k.jsonl'
AUDIT_PATH = PROJECT_DIR / 'v062_audit.json'
ZIP_PATH = PROJECT_DIR / 'fragburden0.3.0.zip'
REPO_DIR = PROJECT_DIR / 'fragburden_src' / 'fragburden'

MODEL_NAME = 'Qwen/Qwen3-4B'
TRANSFORMERS_VERSION = '5.15.1'
TASK_VERSION = 'symbolic_assoc_r1_v062'
SCORER_VERSION = 'assoc_chunked_margin_v062'
POSITION_INTERVENTION = 'query_shift_vs_filler_spread_v1'
PREFILL_CHUNK = 512
TARGET_LENGTHS = [4096, 8192]
FILLER_LANGUAGES = ['en', 'tr']
SEMANTIC_SEEDS = [3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020, 3021, 3022, 3023, 3024, 3025, 3026, 3027, 3028, 3029, 3030, 3031]
POSITION_RANK_SET = [0, 2, 5, 7]
BOOTSTRAP_REPS = 10000
BOOTSTRAP_SEED = 20260828

# Frozen during outcome-independent calibration and reported in the paper.
N_RECORDS = 32
FRAG_KEEP_BY_LANGUAGE = {'en': 2, 'tr': 5}
previous = {'factorial_position_replication': True}

print('FROZEN PAPER SETTINGS')
print('N_RECORDS =', N_RECORDS)
print('FRAG_KEEP_BY_LANGUAGE =', FRAG_KEEP_BY_LANGUAGE)
print('SEMANTIC_SEEDS =', SEMANTIC_SEEDS)


## 2. Install `fragburden` and validated Transformers

In [ ]:
import subprocess, sys

if not ZIP_PATH.exists():
    repo_zip = Path('/content/token-distance-positional-geometry/code/fragburden0.3.0.zip')
    if repo_zip.exists():
        shutil.copy2(repo_zip, ZIP_PATH)
        print('Copied fragburden from cloned repository.')
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_names = [n for n in uploaded if n.endswith('.zip')]
        if not zip_names:
            raise RuntimeError('Upload code/fragburden0.3.0.zip from the repository.')
        shutil.copy2(Path(zip_names[0]), ZIP_PATH)

extract_root = PROJECT_DIR / 'fragburden_src'
if extract_root.exists(): shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(ZIP_PATH), str(extract_root))
assert REPO_DIR.exists() and (REPO_DIR / 'pyproject.toml').exists()
subprocess.check_call([sys.executable,'-m','pip','install','-e',f'{REPO_DIR}[gpu,analysis]'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-U',f'transformers=={TRANSFORMERS_VERSION}','accelerate','bitsandbytes'])
print('Installation complete.')


## 3. Imports and model

In [ ]:
import hashlib, random, math, inspect, sys, gc, time
from dataclasses import dataclass

import numpy as np
import pandas as pd

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch
import torch.nn.functional as F
import transformers

import fragburden
from fragburden.dial import FragmentationDial, verify_monotone
from fragburden.testing import sample_texts

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DynamicCache,
)

assert transformers.__version__ == TRANSFORMERS_VERSION
assert torch.cuda.is_available()

print('fragburden:', fragburden.__file__)
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

load_kwargs = dict(
    quantization_config=bnb,
    device_map='auto',
    attn_implementation='sdpa',
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        **load_kwargs,
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        **load_kwargs,
    )

model.eval()

if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False

backbone = model.model
device = next(model.parameters()).device

if 'position_ids' not in inspect.signature(backbone.forward).parameters:
    raise RuntimeError('Backbone does not expose position_ids.')

print('Loaded:', MODEL_NAME)

## 4. Fragmentation dial gate

In [ ]:
dial = FragmentationDial.from_transformers(
    tokenizer,
    name=MODEL_NAME,
)

cap = dial.capability()

dial_gate = verify_monotone(
    dial,
    sample_texts('en') + sample_texts('tr'),
    levels=(100, 50, 25, 10, 5, 2, 1),
)

print(json.dumps(cap, indent=2))
print(json.dumps(dial_gate, indent=2))

if not cap.get('usable_for_arm_a'):
    raise RuntimeError('No usable fragmentation dial.')

if (
    not dial_gate['monotone_nondecreasing']
    or not dial_gate['all_in_vocab']
):
    raise RuntimeError('Fragmentation dial failed.')

d100 = dial.at(100)

DIALS = {
    language: dial.at(
        FRAG_KEEP_BY_LANGUAGE[language]
    )
    for language in FILLER_LANGUAGES
}

print('DIAL GATE PASSED.')

## 5. Official non-thinking chat shell and digit candidates

In [ ]:
DIGITS = list('1234')

def _normalize_input_ids(obj):
    if hasattr(obj, 'keys') and 'input_ids' in obj:
        obj = obj['input_ids']

    if torch.is_tensor(obj):
        obj = obj.detach().cpu().tolist()

    if (
        isinstance(obj, (list, tuple))
        and len(obj) == 1
        and isinstance(obj[0], (list, tuple))
    ):
        obj = obj[0]

    return [int(x) for x in obj]

def apply_chat_ids(user_content):
    out = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=True,
        return_dict=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return _normalize_input_ids(out)

def render_chat(user_content):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

DIGIT_IDS = {}

for d in DIGITS:
    ids = tokenizer(
        d,
        add_special_tokens=False,
    )['input_ids']

    if len(ids) != 1:
        raise RuntimeError(f'{d} is not one token.')

    DIGIT_IDS[d] = int(ids[0])

if len(set(DIGIT_IDS.values())) != 4:
    raise RuntimeError('Digit tokens are not distinct.')

SHELL_MARKER = 'UNIQUE_USER_CONTENT_MARKER_V061_A93F27'

rendered_marker = render_chat(SHELL_MARKER)

if rendered_marker.count(SHELL_MARKER) != 1:
    raise RuntimeError('Could not isolate user-content shell.')

marker_idx = rendered_marker.index(SHELL_MARKER)

CHAT_PREFIX_TEXT = rendered_marker[:marker_idx]
CHAT_SUFFIX_TEXT = rendered_marker[
    marker_idx + len(SHELL_MARKER):
]

CHAT_PREFIX_IDS = tokenizer(
    CHAT_PREFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

CHAT_SUFFIX_IDS = tokenizer(
    CHAT_SUFFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

def canonical_user_ids(user_content):
    ids_tok = tokenizer(
        user_content,
        add_special_tokens=False,
    )['input_ids']

    ids_dial = d100.pieces_to_ids(
        d100.encode_pieces(user_content)
    )

    if ids_tok != ids_dial:
        raise RuntimeError(
            'Canonical dial IDs disagree with tokenizer.'
        )

    return ids_dial

def verify_chat_shell(user_content):
    user_ids = canonical_user_ids(user_content)

    reconstructed = (
        CHAT_PREFIX_IDS
        + user_ids
        + CHAT_SUFFIX_IDS
    )

    official = apply_chat_ids(user_content)

    if reconstructed != official:
        raise RuntimeError('Chat-shell reconstruction failed.')

    return {
        'official_ids': official,
        'canonical_user_ids': user_ids,
    }

print('Digit IDs:', DIGIT_IDS)
print('Prefix tokens:', len(CHAT_PREFIX_IDS))
print('Suffix tokens:', len(CHAT_SUFFIX_IDS))
print('Chat shell ready.')

## 6. New 32-seed balanced associative-retrieval generator

In [ ]:
@dataclass
class CharSpan:
    start: int
    end: int
    label: str

@dataclass
class AssocInstance:
    filler_language: str
    semantic_seed: int
    trial_index: int
    target_chat_tokens: int
    user_content: str
    spans: list
    target_key: str
    gold_digit: str
    target_record_position: int
    evidence_lines: list
    filler_lines: list
    query_lines: list

def stable_int(*parts):
    blob = '|'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(
        hashlib.sha256(blob).digest()[:8],
        'big',
    )

def stable_rng(*parts):
    return random.Random(stable_int(*parts))

def make_key(semantic_seed, slot):
    value = (
        stable_int(
            'key_v062',
            semantic_seed,
            slot,
            N_RECORDS,
        )
        % 90000
        + 10000
    )

    return f'K{value}'

def gold_for_trial(trial_index):
    return DIGITS[trial_index % 4]

def position_rank_for_trial(trial_index):
    gold_index = trial_index % 4
    replicate = trial_index // 4

    return POSITION_RANK_SET[
        (replicate + gold_index)
        % len(POSITION_RANK_SET)
    ]

def target_position_for_trial(trial_index):
    rank = position_rank_for_trial(trial_index)

    return int(
        round(
            rank
            * (N_RECORDS - 1)
            / 7
        )
    )

FILLER_BANK = {
    'en': {
        'subjects': [
            'The archivist',
            'The technician',
            'The editor',
            'The visitor',
            'The clerk',
            'The gardener',
        ],
        'verbs': [
            'reviewed',
            'checked',
            'organized',
            'recorded',
            'sorted',
            'noted',
        ],
        'objects': [
            'a routine report',
            'the daily schedule',
            'a short memo',
            'the supply list',
            'a maintenance note',
            'the meeting agenda',
        ],
        'settings': [
            'near the window',
            'in the quiet office',
            'before lunch',
            'during the afternoon',
            'at the front desk',
            'after the meeting',
        ],
    },
    'tr': {
        'subjects': [
            'Arşiv görevlisi',
            'Teknisyen',
            'Editör',
            'Ziyaretçi',
            'Görevli',
            'Bahçıvan',
        ],
        'verbs': [
            'inceledi',
            'kontrol etti',
            'düzenledi',
            'kaydetti',
            'sıraladı',
            'not etti',
        ],
        'objects': [
            'sıradan bir raporu',
            'günlük programı',
            'kısa bir notu',
            'malzeme listesini',
            'bakım notunu',
            'toplantı gündemini',
        ],
        'settings': [
            'pencerenin yanında',
            'sessiz ofiste',
            'öğle yemeğinden önce',
            'öğleden sonra',
            'ön masada',
            'toplantıdan sonra',
        ],
    },
}

def filler_sentence(language, idx, semantic_seed):
    bank = FILLER_BANK[language]

    x = idx + 7919 * semantic_seed

    subject = bank['subjects'][x % len(bank['subjects'])]
    x //= len(bank['subjects'])

    verb = bank['verbs'][x % len(bank['verbs'])]
    x //= len(bank['verbs'])

    obj = bank['objects'][x % len(bank['objects'])]
    x //= len(bank['objects'])

    setting = bank['settings'][x % len(bank['settings'])]

    if language == 'en':
        return f'{subject} {verb} {obj} {setting}.'

    return f'{subject} {setting} {obj} {verb}.'

def make_records(semantic_seed, trial_index):
    gold = gold_for_trial(trial_index)

    values = DIGITS * (N_RECORDS // 4)

    gold_idx = values.index(gold)

    values[0], values[gold_idx] = (
        values[gold_idx],
        values[0],
    )

    records = [
        (
            make_key(
                semantic_seed,
                slot,
            ),
            values[slot],
        )
        for slot in range(N_RECORDS)
    ]

    target = records[0]
    others = records[1:]

    stable_rng(
        'records_v062',
        semantic_seed,
        trial_index,
    ).shuffle(others)

    pos = target_position_for_trial(trial_index)

    ordered = others[:]
    ordered.insert(pos, target)

    return ordered, target[0], gold, pos

def base_evidence_query(semantic_seed, trial_index):
    records, target_key, gold, pos = make_records(
        semantic_seed,
        trial_index,
    )

    evidence = [
        'Task: look up the value for the queried key.',
        'Each evidence line has the form KEY=VALUE.',
        'Return only the VALUE associated with the queried KEY.',
        'Do not explain or repeat the key.',
        'EVIDENCE:',
    ]

    evidence += [
        f'{key}={value}'
        for key, value in records
    ]

    query = [
        f'QUERY KEY: {target_key}',
        'Answer with one digit only:',
    ]

    return evidence, query, target_key, gold, pos

def compose_user_content(evidence, filler, query):
    evidence_text = '\n'.join(evidence)
    filler_text = '\n'.join(filler)
    query_text = '\n'.join(query)

    user_content = (
        evidence_text
        + '\n'
        + filler_text
        + '\n'
        + query_text
    )

    filler_start = len(evidence_text) + 1
    query_start = filler_start + len(filler_text) + 1

    spans = [
        CharSpan(
            0,
            filler_start,
            'evidence',
        ),
        CharSpan(
            filler_start,
            query_start,
            'filler',
        ),
        CharSpan(
            query_start,
            len(user_content),
            'query',
        ),
    ]

    return user_content, spans

def build_instance(
    filler_language,
    semantic_seed,
    trial_index,
    target_chat_tokens,
):
    (
        evidence,
        query,
        target_key,
        gold,
        pos,
    ) = base_evidence_query(
        semantic_seed,
        trial_index,
    )

    filler = []

    while True:
        user_content, spans = compose_user_content(
            evidence,
            filler,
            query,
        )

        if len(apply_chat_ids(user_content)) >= target_chat_tokens:
            break

        filler.append(
            filler_sentence(
                filler_language,
                len(filler),
                semantic_seed,
            )
        )

        if len(filler) > 5000:
            raise RuntimeError('Filler loop failed.')

    verify_chat_shell(user_content)

    return AssocInstance(
        filler_language=filler_language,
        semantic_seed=semantic_seed,
        trial_index=trial_index,
        target_chat_tokens=target_chat_tokens,
        user_content=user_content,
        spans=spans,
        target_key=target_key,
        gold_digit=gold,
        target_record_position=pos,
        evidence_lines=evidence,
        filler_lines=filler,
        query_lines=query,
    )

print('Generator ready.')

## 7. Exact gold × target-position balance

In [ ]:
balance_rows = []

for trial_index, semantic_seed in enumerate(SEMANTIC_SEEDS):
    balance_rows.append({
        'trial_index': trial_index,
        'semantic_seed': semantic_seed,
        'gold_digit': gold_for_trial(trial_index),
        'position_rank': position_rank_for_trial(trial_index),
        'target_record_position': target_position_for_trial(trial_index),
    })

balance = pd.DataFrame(balance_rows)
display(balance)

gold_position = pd.crosstab(
    balance['gold_digit'],
    balance['position_rank'],
).reindex(
    index=DIGITS,
    columns=POSITION_RANK_SET,
    fill_value=0,
)

display(gold_position)

if not (gold_position == 2).all().all():
    raise RuntimeError(
        'Every gold×position combination must occur exactly twice.'
    )

print('BALANCE GATE PASSED.')

## 8. Canonical and fragmented source-position construction

In [ ]:
def span_label(start, end, spans):
    best_label = None
    best_overlap = 0

    for s in spans:
        overlap = max(
            0,
            min(end, s.end)
            - max(start, s.start),
        )

        if overlap > best_overlap:
            best_overlap = overlap
            best_label = s.label

    if best_label is None:
        raise RuntimeError(
            f'Chunk [{start},{end}) has no semantic region.'
        )

    return best_label

def _piece_intervals(pieces):
    intervals = []
    pos = 0

    for piece in pieces:
        intervals.append(
            (pos, pos + len(piece))
        )
        pos += len(piece)

    return intervals, pos

def _fractional_local_positions(
    canonical_pieces,
    fragmented_pieces,
):
    if ''.join(canonical_pieces) != ''.join(fragmented_pieces):
        raise RuntimeError(
            'Piece streams reconstruct different text.'
        )

    c_int, c_total = _piece_intervals(canonical_pieces)
    f_int, f_total = _piece_intervals(fragmented_pieces)

    if c_total != f_total:
        raise RuntimeError('Character extents disagree.')

    assignment = []
    c_idx = 0

    for fa, fb in f_int:
        while (
            c_idx + 1 < len(c_int)
            and c_int[c_idx][1] <= fa
        ):
            c_idx += 1

        ca, cb = c_int[c_idx]

        if not (ca <= fa and fb <= cb):
            raise RuntimeError(
                'Fragment crossed a canonical-token boundary.'
            )

        assignment.append(c_idx)

    groups = {}

    for fine_idx, coarse_idx in enumerate(assignment):
        groups.setdefault(coarse_idx, []).append(fine_idx)

    positions = [None] * len(fragmented_pieces)

    for coarse_idx, fine_indices in groups.items():
        m = len(fine_indices)

        for rank, fine_idx in enumerate(fine_indices):
            positions[fine_idx] = (
                coarse_idx
                + rank / m
            )

    arr = np.asarray(positions, dtype=np.float32)

    if (
        len(arr) > 1
        and not np.all(np.diff(arr) > 0)
    ):
        raise RuntimeError(
            'Source-aligned local positions are not strictly increasing.'
        )

    return arr.tolist()

def build_user_encoding(inst):
    canonical_ids = []
    fragmented_ids = []

    canonical_labels = []
    fragmented_labels = []

    fragmented_source_positions = []

    canonical_cursor = 0
    dfrag = DIALS[inst.filler_language]

    for chunk, (a, b) in d100.pre_tokenize(inst.user_content):
        label = span_label(a, b, inst.spans)

        c_pieces = d100.encode_chunk_pieces(chunk)
        c_ids = d100.pieces_to_ids(c_pieces)

        canonical_ids.extend(c_ids)
        canonical_labels.extend([label] * len(c_ids))

        if label == 'filler':
            f_pieces = dfrag.encode_chunk_pieces(chunk)
            f_ids = d100.pieces_to_ids(f_pieces)

            local = _fractional_local_positions(
                c_pieces,
                f_pieces,
            )

            src = [
                canonical_cursor + x
                for x in local
            ]

        else:
            f_ids = c_ids
            src = [
                float(x)
                for x in range(
                    canonical_cursor,
                    canonical_cursor + len(c_ids),
                )
            ]

        fragmented_ids.extend(f_ids)
        fragmented_labels.extend([label] * len(f_ids))
        fragmented_source_positions.extend(src)

        canonical_cursor += len(c_ids)

    whole = canonical_user_ids(inst.user_content)

    if canonical_ids != whole:
        raise RuntimeError(
            'Chunkwise canonical encoding mismatch.'
        )

    decoded = tokenizer.decode(
        fragmented_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    if decoded != inst.user_content:
        raise RuntimeError(
            'Fragmented IDs do not decode to identical user content.'
        )

    return {
        'canonical_ids': canonical_ids,
        'fragmented_ids': fragmented_ids,
        'canonical_labels': canonical_labels,
        'fragmented_labels': fragmented_labels,
        'fragmented_source_positions': np.asarray(
            fragmented_source_positions,
            dtype=np.float32,
        ),
    }

print('Encoding helpers ready.')

## 9. Six position maps: source → query-shift → filler-spread

The two-step expansion path is:

`source geometry → query/readout shift only → filler positional spreading`

For canonical IDs:

`CC → CQ → CE`

For fragmented IDs:

`FC → FQ → FE`

`CQ` and `CE` have the same token IDs, query positions, suffix positions, and global evidence→readout distance.  
They differ only in filler-token positions.

Likewise for `FQ` and `FE`.

In [ ]:
def hash_ids(ids):
    return hashlib.sha256(
        np.asarray(
            ids,
            dtype=np.int64,
        ).tobytes()
    ).hexdigest()[:24]

def hash_positions(pos):
    return hashlib.sha256(
        np.asarray(
            pos,
            dtype=np.float32,
        ).tobytes()
    ).hexdigest()[:24]

def full_gap_diagnostics(
    positions,
    prefix_len,
    labels,
):
    labels = np.asarray(labels)

    evidence_idx = np.where(
        labels == 'evidence'
    )[0]

    filler_idx = np.where(
        labels == 'filler'
    )[0]

    query_idx = np.where(
        labels == 'query'
    )[0]

    if not (
        len(evidence_idx)
        and len(filler_idx)
        and len(query_idx)
    ):
        raise RuntimeError(
            'Missing evidence/filler/query region.'
        )

    evidence_last = float(
        positions[
            prefix_len + evidence_idx[-1]
        ]
    )

    filler_first = float(
        positions[
            prefix_len + filler_idx[0]
        ]
    )

    filler_last = float(
        positions[
            prefix_len + filler_idx[-1]
        ]
    )

    query_first = float(
        positions[
            prefix_len + query_idx[0]
        ]
    )

    query_last = float(
        positions[
            prefix_len + query_idx[-1]
        ]
    )

    readout = float(
        positions[-1]
    )

    return {
        'evidence_query': query_first - evidence_last,
        'evidence_readout': readout - evidence_last,
        'filler_span': filler_last - filler_first,
        'filler_query_gap': query_first - filler_last,
        'query_readout': readout - query_last,
    }

def assemble_geometry(inst):
    shell = verify_chat_shell(
        inst.user_content
    )

    enc = build_user_encoding(inst)

    c_user = enc['canonical_ids']
    f_user = enc['fragmented_ids']

    c_labels = np.asarray(
        enc['canonical_labels']
    )

    f_labels = np.asarray(
        enc['fragmented_labels']
    )

    prefix_len = len(CHAT_PREFIX_IDS)
    suffix_len = len(CHAT_SUFFIX_IDS)

    c_full_ids = (
        CHAT_PREFIX_IDS
        + c_user
        + CHAT_SUFFIX_IDS
    )

    f_full_ids = (
        CHAT_PREFIX_IDS
        + f_user
        + CHAT_SUFFIX_IDS
    )

    if c_full_ids != shell['official_ids']:
        raise RuntimeError(
            'Canonical full IDs disagree with official chat.'
        )

    # --- Source geometries ---
    cc_pos = np.arange(
        len(c_full_ids),
        dtype=np.float32,
    )

    prefix_pos = np.arange(
        prefix_len,
        dtype=np.float32,
    )

    canonical_suffix_start = (
        prefix_len
        + len(c_user)
    )

    canonical_suffix_pos = np.arange(
        canonical_suffix_start,
        canonical_suffix_start + suffix_len,
        dtype=np.float32,
    )

    fc_user_pos = (
        prefix_len
        + enc[
            'fragmented_source_positions'
        ]
    )

    fc_pos = np.concatenate(
        [
            prefix_pos,
            fc_user_pos,
            canonical_suffix_pos,
        ]
    ).astype(np.float32)

    # --- Fully expanded geometry (natural fragmented positions) ---
    fe_pos = np.arange(
        len(f_full_ids),
        dtype=np.float32,
    )

    # Region indices.
    c_filler_idx = np.where(
        c_labels == 'filler'
    )[0]

    c_query_idx = np.where(
        c_labels == 'query'
    )[0]

    f_filler_idx = np.where(
        f_labels == 'filler'
    )[0]

    f_query_idx = np.where(
        f_labels == 'query'
    )[0]

    if not (
        len(c_filler_idx)
        and len(c_query_idx)
        and len(f_filler_idx)
        and len(f_query_idx)
    ):
        raise RuntimeError(
            'Missing canonical/fragmented regions.'
        )

    c_query_first = int(
        c_query_idx[0]
    )

    f_query_first = int(
        f_query_idx[0]
    )

    query_offset = (
        f_query_first
        - c_query_first
    )

    if query_offset <= 0:
        raise RuntimeError(
            'Fragmentation did not create a positive query-position offset.'
        )

    expanded_suffix_start = (
        prefix_len
        + len(f_user)
    )

    expanded_suffix_pos = np.arange(
        expanded_suffix_start,
        expanded_suffix_start + suffix_len,
        dtype=np.float32,
    )

    # --- CQ: canonical IDs; keep evidence+filler positions canonical;
    # shift only query + assistant suffix to expanded absolute positions. ---
    cq_user_pos = np.arange(
        len(c_user),
        dtype=np.float32,
    )

    cq_user_pos[
        c_query_idx
    ] = (
        c_query_idx.astype(
            np.float32
        )
        + query_offset
    )

    cq_pos = np.concatenate(
        [
            prefix_pos,
            prefix_len + cq_user_pos,
            expanded_suffix_pos,
        ]
    ).astype(np.float32)

    # --- CE: same query/suffix positions as CQ, plus filler spread. ---
    ce_user_pos = cq_user_pos.copy()

    c_filler_first = int(
        c_filler_idx[0]
    )

    c_filler_n = len(
        c_filler_idx
    )

    ce_user_pos[
        c_filler_idx
    ] = (
        c_filler_first
        + np.arange(
            c_filler_n,
            dtype=np.float32,
        )
        * (
            f_query_first
            - c_filler_first
        )
        / c_filler_n
    )

    ce_pos = np.concatenate(
        [
            prefix_pos,
            prefix_len + ce_user_pos,
            expanded_suffix_pos,
        ]
    ).astype(np.float32)

    # --- FQ: fragmented IDs; keep source-aligned evidence+filler positions;
    # shift only query + assistant suffix to expanded absolute positions. ---
    fq_user_pos = enc[
        'fragmented_source_positions'
    ].copy()

    fq_user_pos[
        f_query_idx
    ] = (
        fq_user_pos[
            f_query_idx
        ]
        + query_offset
    )

    fq_pos = np.concatenate(
        [
            prefix_pos,
            prefix_len + fq_user_pos,
            expanded_suffix_pos,
        ]
    ).astype(np.float32)

    maps = {
        'CC': {
            'condition': 'canonical_ids_source_pos',
            'ids': c_full_ids,
            'positions': cc_pos,
            'labels': c_labels,
        },
        'CQ': {
            'condition': 'canonical_ids_query_shift_pos',
            'ids': c_full_ids,
            'positions': cq_pos,
            'labels': c_labels,
        },
        'CE': {
            'condition': 'canonical_ids_filler_spread_pos',
            'ids': c_full_ids,
            'positions': ce_pos,
            'labels': c_labels,
        },
        'FC': {
            'condition': 'fragmented_ids_source_pos',
            'ids': f_full_ids,
            'positions': fc_pos,
            'labels': f_labels,
        },
        'FQ': {
            'condition': 'fragmented_ids_query_shift_pos',
            'ids': f_full_ids,
            'positions': fq_pos,
            'labels': f_labels,
        },
        'FE': {
            'condition': 'fragmented_ids_filler_spread_pos',
            'ids': f_full_ids,
            'positions': fe_pos,
            'labels': f_labels,
        },
    }

    for key, item in maps.items():
        p = item['positions']

        if len(item['ids']) != len(p):
            raise RuntimeError(
                f'{key}: IDs/positions length mismatch.'
            )

        if (
            len(p) > 1
            and not np.all(np.diff(p) > 0)
        ):
            bad = np.where(
                np.diff(p) <= 0
            )[0][:10]

            raise RuntimeError(
                f'{key}: positions not strictly increasing: {bad.tolist()}'
            )

        item['ids_hash'] = hash_ids(
            item['ids']
        )

        item['position_hash'] = hash_positions(
            p
        )

        item['gaps'] = full_gap_diagnostics(
            p,
            prefix_len,
            item['labels'],
        )

        item['realized_tokens'] = len(
            item['ids']
        )

        item['position_max'] = float(
            p[-1]
        )

    # Same-ID invariants.
    if not (
        maps['CC']['ids_hash']
        == maps['CQ']['ids_hash']
        == maps['CE']['ids_hash']
    ):
        raise RuntimeError(
            'CC/CQ/CE must have identical canonical IDs.'
        )

    if not (
        maps['FC']['ids_hash']
        == maps['FQ']['ids_hash']
        == maps['FE']['ids_hash']
    ):
        raise RuntimeError(
            'FC/FQ/FE must have identical fragmented IDs.'
        )

    # Source geometry.
    if abs(
        maps['CC']['gaps']['evidence_readout']
        - maps['FC']['gaps']['evidence_readout']
    ) > 1e-3:
        raise RuntimeError(
            'CC and FC source geometry differs.'
        )

    # Query-shift geometry: same global gap and same filler span.
    if abs(
        maps['CQ']['gaps']['evidence_readout']
        - maps['FQ']['gaps']['evidence_readout']
    ) > 1e-3:
        raise RuntimeError(
            'CQ and FQ global geometry differs.'
        )

    if abs(
        maps['CQ']['gaps']['filler_span']
        - maps['CC']['gaps']['filler_span']
    ) > 1e-3:
        raise RuntimeError(
            'CQ unexpectedly changed canonical filler span.'
        )

    if abs(
        maps['FQ']['gaps']['filler_span']
        - maps['FC']['gaps']['filler_span']
    ) > 1e-3:
        raise RuntimeError(
            'FQ unexpectedly changed fragmented source filler span.'
        )

    # Full-spread geometry: same global gap as query-shift,
    # but much larger filler positional span.
    for a, b in [
        ('CQ', 'CE'),
        ('FQ', 'FE'),
    ]:
        if abs(
            maps[a]['gaps']['evidence_readout']
            - maps[b]['gaps']['evidence_readout']
        ) > 1e-3:
            raise RuntimeError(
                f'{a}/{b} global evidence→readout geometry differs.'
            )

        if abs(
            maps[a]['position_max']
            - maps[b]['position_max']
        ) > 1e-3:
            raise RuntimeError(
                f'{a}/{b} readout absolute position differs.'
            )

        if not (
            maps[b]['gaps']['filler_span']
            > maps[a]['gaps']['filler_span']
        ):
            raise RuntimeError(
                f'{a}->{b} did not spread filler positions.'
            )

    if not (
        maps['CQ']['gaps']['evidence_readout']
        > maps['CC']['gaps']['evidence_readout']
    ):
        raise RuntimeError(
            'Query shift did not increase global evidence→readout distance.'
        )

    return maps

print('Six-condition geometry assembly ready.')

## 10. T4-safe scorer with raw candidate logit margins

In [ ]:
def _sequential_chunk_mask(start, end, device):
    q = torch.arange(
        start,
        end,
        device=device,
    )[:, None]

    k = torch.arange(
        0,
        end,
        device=device,
    )[None, :]

    return (
        k <= q
    )[None, None, :, :]

def finish_scores(h_last, gold_digit):
    logits = model.lm_head(
        h_last
    ).float()[0]

    candidate_ids = torch.tensor(
        [
            DIGIT_IDS[d]
            for d in DIGITS
        ],
        dtype=torch.long,
        device=logits.device,
    )

    candidate_logits = logits.index_select(
        0,
        candidate_ids,
    )

    candidate_logp = F.log_softmax(
        candidate_logits,
        dim=-1,
    )

    candidate_prob = candidate_logp.exp()

    gold_idx = DIGITS.index(
        gold_digit
    )

    distractor_indices = [
        i
        for i in range(len(DIGITS))
        if i != gold_idx
    ]

    best_distractor_idx = max(
        distractor_indices,
        key=lambda i: float(
            candidate_logits[i].item()
        ),
    )

    gold_margin = float(
        (
            candidate_logits[gold_idx]
            - candidate_logits[best_distractor_idx]
        ).item()
    )

    pred_idx = int(
        torch.argmax(
            candidate_logits
        ).item()
    )

    pred = DIGITS[pred_idx]

    full_argmax_id = int(
        torch.argmax(logits).item()
    )

    full_argmax_digit = next(
        (
            d
            for d, tok_id in DIGIT_IDS.items()
            if tok_id == full_argmax_id
        ),
        None,
    )

    result = {
        'predicted_digit': pred,
        'correct': int(
            pred == gold_digit
        ),
        'gold_logit_margin': gold_margin,
        'gold_logit': float(
            candidate_logits[gold_idx].item()
        ),
        'best_distractor_digit': DIGITS[
            best_distractor_idx
        ],
        'best_distractor_logit': float(
            candidate_logits[
                best_distractor_idx
            ].item()
        ),
        'forced_choice_nll': float(
            -candidate_logp[
                gold_idx
            ].item()
        ),
        'gold_candidate_probability': float(
            candidate_prob[
                gold_idx
            ].item()
        ),
        'full_argmax_id': full_argmax_id,
        'full_argmax_text': tokenizer.decode(
            [full_argmax_id],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ),
        'full_argmax_digit': full_argmax_digit,
        'full_argmax_is_gold': int(
            full_argmax_digit == gold_digit
        ),
        'candidate_logits': {
            d: float(
                candidate_logits[i].item()
            )
            for i, d in enumerate(DIGITS)
        },
        'candidate_probabilities': {
            d: float(
                candidate_prob[i].item()
            )
            for i, d in enumerate(DIGITS)
        },
    }

    del (
        logits,
        candidate_ids,
        candidate_logits,
        candidate_logp,
        candidate_prob,
    )

    return result

def score_chunked(
    input_ids,
    position_ids,
    gold_digit,
    chunk_size=PREFILL_CHUNK,
):
    if len(input_ids) != len(position_ids):
        raise ValueError('IDs/positions length mismatch.')

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    ids_all = torch.tensor(
        input_ids,
        dtype=torch.long,
        device=device,
    )

    pos_all = torch.tensor(
        position_ids,
        dtype=torch.float32,
        device=device,
    )

    cache = DynamicCache(
        config=model.config
    )

    last_hidden = None

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        n = ids_all.numel()

        for start in range(
            0,
            n,
            chunk_size,
        ):
            end = min(
                start + chunk_size,
                n,
            )

            ids_chunk = ids_all[
                start:end
            ].unsqueeze(0)

            pos_chunk = pos_all[
                start:end
            ].unsqueeze(0)

            causal = _sequential_chunk_mask(
                start,
                end,
                device,
            )

            out = backbone(
                input_ids=ids_chunk,
                attention_mask={
                    'full_attention': causal
                },
                position_ids=pos_chunk,
                past_key_values=cache,
                use_cache=True,
                return_dict=True,
            )

            cache = out.past_key_values

            if end == n:
                last_hidden = out.last_hidden_state[
                    :, -1, :
                ]

            del (
                ids_chunk,
                pos_chunk,
                causal,
                out,
            )

        scored = finish_scores(
            last_hidden,
            gold_digit,
        )

    torch.cuda.synchronize()

    scored['seconds'] = (
        time.perf_counter() - t0
    )

    scored['peak_cuda_gb'] = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    scored['scorer_version'] = SCORER_VERSION

    del (
        ids_all,
        pos_all,
        cache,
        last_hidden,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return scored

def flatten_score(scored):
    out = {
        k: v
        for k, v in scored.items()
        if k not in {
            'candidate_logits',
            'candidate_probabilities',
        }
    }

    for d, v in scored['candidate_logits'].items():
        out[f'logit_{d}'] = v

    for d, v in scored['candidate_probabilities'].items():
        out[f'prob_{d}'] = v

    return out

print('Margin scorer ready.')

## 11. Structural engineering gate at ~1K

In [ ]:
gate_inst = build_instance(
    filler_language='en',
    semantic_seed=3999,
    trial_index=7,
    target_chat_tokens=1024,
)

maps = assemble_geometry(gate_inst)

for key in ['CC', 'CQ', 'CE', 'FC', 'FQ', 'FE']:
    item = maps[key]

    print(
        key,
        {
            'condition': item['condition'],
            'tokens': item['realized_tokens'],
            'position_max': item['position_max'],
            'evidence_readout_gap': item['gaps']['evidence_readout'],
            'filler_span': item['gaps']['filler_span'],
            'filler_query_gap': item['gaps']['filler_query_gap'],
            'ids_hash': item['ids_hash'],
            'position_hash': item['position_hash'],
        },
    )

gate_scores = {}

for key in ['CC', 'CQ', 'CE', 'FC', 'FQ', 'FE']:
    gate_scores[key] = score_chunked(
        maps[key]['ids'],
        maps[key]['positions'],
        gate_inst.gold_digit,
    )

    print(
        key,
        {
            'pred': gate_scores[key]['predicted_digit'],
            'margin': gate_scores[key]['gold_logit_margin'],
            'nll': gate_scores[key]['forced_choice_nll'],
            'full': gate_scores[key]['full_argmax_text'],
            'peak_gb': gate_scores[key]['peak_cuda_gb'],
        },
    )

    if not (
        np.isfinite(
            gate_scores[key]['gold_logit_margin']
        )
        and np.isfinite(
            gate_scores[key]['forced_choice_nll']
        )
    ):
        raise RuntimeError(
            f'Non-finite engineering score in {key}.'
        )

print('STRUCTURAL ENGINEERING GATE PASSED.')

## 12. Resumable JSONL utilities

In [ ]:
def load_jsonl(path):
    rows = []

    if path.exists():
        with path.open(
            'r',
            encoding='utf-8',
        ) as f:
            for line in f:
                line = line.strip()

                if line:
                    rows.append(
                        json.loads(line)
                    )

    return rows

def completed_ids(path):
    return {
        row['result_id']
        for row in load_jsonl(path)
        if 'result_id' in row
    }

def make_result_id(meta):
    blob = json.dumps(
        meta,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')

    return hashlib.sha256(
        blob
    ).hexdigest()[:24]

def append_jsonl(path, row):
    with path.open(
        'a',
        encoding='utf-8',
    ) as f:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + '\n'
        )
        f.flush()

# Phase A — prospective six-condition localization run

32 trials × 2 filler languages × 2 lengths × 6 conditions = **768 rows**.

Condition order is deterministically randomized inside every base instance.

In [ ]:
CONDITIONS = ['CC', 'CQ', 'CE', 'FC', 'FQ', 'FE']

done = completed_ids(
    RESULTS_PATH
)

for filler_language in FILLER_LANGUAGES:
    for target_chat_tokens in TARGET_LENGTHS:
        for trial_index, semantic_seed in enumerate(
            SEMANTIC_SEEDS
        ):
            inst = build_instance(
                filler_language=filler_language,
                semantic_seed=semantic_seed,
                trial_index=trial_index,
                target_chat_tokens=target_chat_tokens,
            )

            maps = assemble_geometry(inst)

            order = CONDITIONS[:]

            stable_rng(
                'condition_order_v062',
                filler_language,
                target_chat_tokens,
                semantic_seed,
                trial_index,
            ).shuffle(order)

            evidence_text = '\n'.join(
                inst.evidence_lines
            )

            query_text = '\n'.join(
                inst.query_lines
            )

            for order_index, key in enumerate(order):
                item = maps[key]

                meta = {
                    'phase': 'geometry_localization_4k8k',
                    'model': MODEL_NAME,
                    'task_version': TASK_VERSION,
                    'scorer_version': SCORER_VERSION,
                    'position_intervention': POSITION_INTERVENTION,
                    'filler_language': filler_language,
                    'target_chat_tokens': target_chat_tokens,
                    'semantic_seed': semantic_seed,
                    'trial_index': trial_index,
                    'n_records': N_RECORDS,
                    'geometry_code': key,
                    'condition': item['condition'],
                    'frag_keep': (
                        None
                        if key in {'CC', 'CQ', 'CE'}
                        else FRAG_KEEP_BY_LANGUAGE[
                            filler_language
                        ]
                    ),
                }

                rid = make_result_id(meta)

                if rid in done:
                    continue

                scored = score_chunked(
                    item['ids'],
                    item['positions'],
                    inst.gold_digit,
                )

                row = {
                    'result_id': rid,
                    **meta,
                    'condition_order_index': order_index,
                    'gold_digit': inst.gold_digit,
                    'target_key': inst.target_key,
                    'position_rank': position_rank_for_trial(
                        trial_index
                    ),
                    'target_record_position': inst.target_record_position,
                    'realized_full_tokens': item[
                        'realized_tokens'
                    ],
                    'position_max': item[
                        'position_max'
                    ],
                    'evidence_query_position_gap': item[
                        'gaps'
                    ][
                        'evidence_query'
                    ],
                    'evidence_readout_position_gap': item[
                        'gaps'
                    ][
                        'evidence_readout'
                    ],
                    'filler_position_span': item[
                        'gaps'
                    ][
                        'filler_span'
                    ],
                    'filler_query_position_gap': item[
                        'gaps'
                    ][
                        'filler_query_gap'
                    ],
                    'ids_hash': item[
                        'ids_hash'
                    ],
                    'position_hash': item[
                        'position_hash'
                    ],
                    'prompt_sha256': hashlib.sha256(
                        inst.user_content.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'evidence_sha256': hashlib.sha256(
                        evidence_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'query_sha256': hashlib.sha256(
                        query_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
                    **flatten_score(scored),
                }

                append_jsonl(
                    RESULTS_PATH,
                    row,
                )

                done.add(rid)

                print(
                    f"fill={filler_language} "
                    f"L={target_chat_tokens:5d} "
                    f"trial={trial_index:02d} "
                    f"{key} "
                    f"gold={inst.gold_digit} "
                    f"pred={scored['predicted_digit']} "
                    f"margin={scored['gold_logit_margin']:+8.3f} "
                    f"NLL={scored['forced_choice_nll']:.5f} "
                    f"n={item['realized_tokens']:5d} "
                    f"ERgap={item['gaps']['evidence_readout']:8.1f} "
                    f"Fspan={item['gaps']['filler_span']:8.1f} "
                    f"peak={scored['peak_cuda_gb']:.2f}GB"
                )

print('Localization run complete:', RESULTS_PATH)

## 13. Pairing and six-condition geometry gates

In [ ]:
df = pd.DataFrame(
    load_jsonl(
        RESULTS_PATH
    )
)

base_cols = [
    'filler_language',
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

condition_counts = df.groupby(
    base_cols
)['geometry_code'].nunique()

if (condition_counts != 6).any():
    raise RuntimeError(
        'Some base instances do not have all six geometry conditions.'
    )

if (
    df.groupby(
        base_cols
    )[
        'prompt_sha256'
    ].nunique()
    != 1
).any():
    raise RuntimeError(
        'Prompt text differs across geometry conditions.'
    )

pair = df.pivot_table(
    index=base_cols,
    columns='geometry_code',
    values=[
        'ids_hash',
        'evidence_readout_position_gap',
        'filler_position_span',
        'position_max',
    ],
    aggfunc='first',
)

# Same IDs within each tokenization family.
if not (
    (
        pair['ids_hash']['CC']
        == pair['ids_hash']['CQ']
    )
    & (
        pair['ids_hash']['CQ']
        == pair['ids_hash']['CE']
    )
).all():
    raise RuntimeError(
        'CC/CQ/CE token IDs differ.'
    )

if not (
    (
        pair['ids_hash']['FC']
        == pair['ids_hash']['FQ']
    )
    & (
        pair['ids_hash']['FQ']
        == pair['ids_hash']['FE']
    )
).all():
    raise RuntimeError(
        'FC/FQ/FE token IDs differ.'
    )

# Source geometry is matched across tokenization.
if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CC']
        - pair[
            'evidence_readout_position_gap'
        ]['FC']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CC/FC source geometry differs.'
    )

# Query-shift geometry is matched across tokenization.
if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CQ']
        - pair[
            'evidence_readout_position_gap'
        ]['FQ']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CQ/FQ global geometry differs.'
    )

# Spread geometry is matched across tokenization.
if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CE']
        - pair[
            'evidence_readout_position_gap'
        ]['FE']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CE/FE global geometry differs.'
    )

# Within each ID family, query-shift and spread conditions have
# identical query/readout geometry and differ only in filler positions.
for q_code, e_code in [
    ('CQ', 'CE'),
    ('FQ', 'FE'),
]:
    if (
        np.abs(
            pair[
                'evidence_readout_position_gap'
            ][q_code]
            - pair[
                'evidence_readout_position_gap'
            ][e_code]
        ).max()
        > 1e-3
    ):
        raise RuntimeError(
            f'{q_code}/{e_code} global gap differs.'
        )

    if (
        np.abs(
            pair[
                'position_max'
            ][q_code]
            - pair[
                'position_max'
            ][e_code]
        ).max()
        > 1e-3
    ):
        raise RuntimeError(
            f'{q_code}/{e_code} readout position differs.'
        )

    if not (
        pair[
            'filler_position_span'
        ][e_code]
        > pair[
            'filler_position_span'
        ][q_code]
    ).all():
        raise RuntimeError(
            f'{q_code}->{e_code} did not spread filler.'
        )

task_pair_cols = [
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

if (
    df.groupby(
        task_pair_cols
    )['evidence_sha256'].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task evidence differs across filler languages.'
    )

if (
    df.groupby(
        task_pair_cols
    )['query_sha256'].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task query differs across filler languages.'
    )

print(
    'PAIRING / SAME-ID / QUERY-MATCH / FILLER-SPREAD GATES PASSED.'
)

## 14. Performance summary

In [ ]:
performance = df.groupby(
    [
        'filler_language',
        'target_chat_tokens',
        'geometry_code',
        'condition',
    ],
    as_index=False,
).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
    mean_margin=('gold_logit_margin', 'mean'),
    median_margin=('gold_logit_margin', 'median'),
    mean_forced_nll=('forced_choice_nll', 'mean'),
    mean_gold_probability=(
        'gold_candidate_probability',
        'mean',
    ),
    full_argmax_gold_rate=(
        'full_argmax_is_gold',
        'mean',
    ),
    mean_tokens=('realized_full_tokens', 'mean'),
    mean_position_max=('position_max', 'mean'),
    mean_evidence_readout_gap=(
        'evidence_readout_position_gap',
        'mean',
    ),
    max_peak_cuda_gb=('peak_cuda_gb', 'max'),
)

display(performance)

# Geometry-localization effects

## 15. Paired logit-margin contrasts

All effects are defined so **positive means better gold-vs-distractor separation**.

Query/readout displacement:

- `query_shift_canonical = CQ - CC`
- `query_shift_fragmented = FQ - FC`

Incremental filler positional spreading, with query/readout fixed:

- `filler_spread_canonical = CE - CQ`
- `filler_spread_fragmented = FE - FQ`

Total source→fully-expanded effect:

- `total_expansion_canonical = CE - CC`
- `total_expansion_fragmented = FE - FC`

Tokenization effects under matched geometries:

- `representation_source = FC - CC`
- `representation_query_shift = FQ - CQ`
- `representation_spread = FE - CE`

In [ ]:
wide = df.pivot_table(
    index=base_cols,
    columns='geometry_code',
    values=[
        'gold_logit_margin',
        'forced_choice_nll',
        'correct',
    ],
    aggfunc='first',
).reset_index()

for code_name in [
    'CC',
    'CQ',
    'CE',
    'FC',
    'FQ',
    'FE',
]:
    if (
        'gold_logit_margin',
        code_name,
    ) not in wide.columns:
        raise RuntimeError(
            f'Missing {code_name} margin column.'
        )

effects = pd.DataFrame({
    'filler_language': wide[
        ('filler_language', '')
    ],
    'target_chat_tokens': wide[
        ('target_chat_tokens', '')
    ],
    'semantic_seed': wide[
        ('semantic_seed', '')
    ],
    'trial_index': wide[
        ('trial_index', '')
    ],
})

CC = wide[
    ('gold_logit_margin', 'CC')
].to_numpy(float)

CQ = wide[
    ('gold_logit_margin', 'CQ')
].to_numpy(float)

CE = wide[
    ('gold_logit_margin', 'CE')
].to_numpy(float)

FC = wide[
    ('gold_logit_margin', 'FC')
].to_numpy(float)

FQ = wide[
    ('gold_logit_margin', 'FQ')
].to_numpy(float)

FE = wide[
    ('gold_logit_margin', 'FE')
].to_numpy(float)

effects[
    'query_shift_canonical'
] = CQ - CC

effects[
    'query_shift_fragmented'
] = FQ - FC

effects[
    'filler_spread_canonical'
] = CE - CQ

effects[
    'filler_spread_fragmented'
] = FE - FQ

effects[
    'total_expansion_canonical'
] = CE - CC

effects[
    'total_expansion_fragmented'
] = FE - FC

effects[
    'representation_source'
] = FC - CC

effects[
    'representation_query_shift'
] = FQ - CQ

effects[
    'representation_spread'
] = FE - CE

effects[
    'query_shift_interaction'
] = (
    (FQ - FC)
    - (CQ - CC)
)

effects[
    'filler_spread_interaction'
] = (
    (FE - FQ)
    - (CE - CQ)
)

for code_name, arr in [
    ('CC', CC),
    ('CQ', CQ),
    ('CE', CE),
    ('FC', FC),
    ('FQ', FQ),
    ('FE', FE),
]:
    effects[
        f'{code_name}_margin'
    ] = arr

display(effects.head(20))

## 16. Cluster bootstrap and cell-wise localization

In [ ]:
EFFECT_NAMES = [
    'query_shift_canonical',
    'query_shift_fragmented',
    'filler_spread_canonical',
    'filler_spread_fragmented',
    'total_expansion_canonical',
    'total_expansion_fragmented',
    'representation_source',
    'representation_query_shift',
    'representation_spread',
    'query_shift_interaction',
    'filler_spread_interaction',
]

def bootstrap_mean_ci(
    values,
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    x = np.asarray(
        pd.Series(values).dropna(),
        dtype=float,
    )

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(x),
        size=(n_boot, len(x)),
    )

    boot = x[idx].mean(axis=1)

    return (
        float(x.mean()),
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975)),
    )

def cluster_bootstrap_mean_ci(
    frame,
    value_col,
    cluster_col='trial_index',
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    clean = frame[
        [cluster_col, value_col]
    ].dropna()

    clusters = np.asarray(
        sorted(
            clean[cluster_col].unique()
        )
    )

    grouped = {
        c: clean.loc[
            clean[cluster_col] == c,
            value_col,
        ].to_numpy(dtype=float)
        for c in clusters
    }

    observed = float(
        clean[value_col].mean()
    )

    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)

    for b in range(n_boot):
        sampled = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )

        vals = np.concatenate(
            [grouped[c] for c in sampled]
        )

        boot[b] = vals.mean()

    return (
        observed,
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975)),
    )

cell_rows = []

for (
    filler_language,
    target_chat_tokens
), g in effects.groupby(
    [
        'filler_language',
        'target_chat_tokens',
    ]
):
    for effect in EFFECT_NAMES:
        mean, lo, hi = bootstrap_mean_ci(
            g[effect],
            seed=(
                BOOTSTRAP_SEED
                + stable_int(
                    filler_language,
                    target_chat_tokens,
                    effect,
                )
                % 100000
            ),
        )

        cell_rows.append({
            'filler_language': filler_language,
            'target_chat_tokens': int(
                target_chat_tokens
            ),
            'effect': effect,
            'n': len(g),
            'mean': mean,
            'ci_lo': lo,
            'ci_hi': hi,
            'positive_n': int(
                (g[effect] > 0).sum()
            ),
            'negative_n': int(
                (g[effect] < 0).sum()
            ),
        })

cell_summary = pd.DataFrame(
    cell_rows
)

display(cell_summary)

pooled_rows = []

for effect in EFFECT_NAMES:
    mean, lo, hi = cluster_bootstrap_mean_ci(
        effects,
        effect,
        seed=(
            BOOTSTRAP_SEED
            + stable_int(
                'pooled',
                effect,
            )
            % 100000
        ),
    )

    pooled_rows.append({
        'effect': effect,
        'n_rows': len(effects),
        'n_trial_clusters': effects[
            'trial_index'
        ].nunique(),
        'mean': mean,
        'ci_lo': lo,
        'ci_hi': hi,
        'positive_n': int(
            (effects[effect] > 0).sum()
        ),
        'negative_n': int(
            (effects[effect] < 0).sum()
        ),
    })

pooled_summary = pd.DataFrame(
    pooled_rows
)

display(pooled_summary)

for _, row in pooled_summary.iterrows():
    print(
        f"{row['effect']:<42} "
        f"mean={row['mean']:+.4f} "
        f"95% cluster-CI=[{row['ci_lo']:+.4f},{row['ci_hi']:+.4f}] "
        f"positive={int(row['positive_n'])}/{int(row['n_rows'])}"
    )

## 17. Leave-one-trial-out influence audit

The v0.6.1 total expansion effect was leave-one-trial-out sign-stable.  
This follow-up applies the same audit separately to query displacement and filler spreading.

In [ ]:
loo_rows = []

for left_out in sorted(
    effects['trial_index'].unique()
):
    g = effects[
        effects['trial_index']
        != left_out
    ]

    row = {
        'left_out_trial': int(left_out),
    }

    for effect in EFFECT_NAMES:
        row[effect] = float(
            g[effect].mean()
        )

    loo_rows.append(row)

loo = pd.DataFrame(
    loo_rows
)

display(loo)

loo_summary = []

for effect in EFFECT_NAMES:
    loo_summary.append({
        'effect': effect,
        'full_mean': float(
            effects[effect].mean()
        ),
        'loo_min': float(
            loo[effect].min()
        ),
        'loo_max': float(
            loo[effect].max()
        ),
        'loo_sign_stable_positive': bool(
            (
                loo[effect] > 0
            ).all()
        ),
        'loo_sign_stable_negative': bool(
            (
                loo[effect] < 0
            ).all()
        ),
    })

loo_summary = pd.DataFrame(
    loo_summary
)

display(loo_summary)

## 18. Position-rank and language×length diagnostics

These are descriptive generality checks and are not used to select a favorable subset.

In [ ]:
meta = balance[
    [
        'trial_index',
        'position_rank',
        'target_record_position',
        'gold_digit',
    ]
]

effects_meta = effects.merge(
    meta,
    on='trial_index',
    how='left',
)

rank_summary = effects_meta.groupby(
    'position_rank',
    as_index=False,
)[
    [
        'query_shift_canonical',
        'query_shift_fragmented',
        'filler_spread_canonical',
        'filler_spread_fragmented',
        'total_expansion_canonical',
        'total_expansion_fragmented',
        'representation_source',
    ]
].mean()

display(rank_summary)

cell_wide = cell_summary.pivot_table(
    index=[
        'filler_language',
        'target_chat_tokens',
    ],
    columns='effect',
    values='mean',
    aggfunc='first',
).reset_index()

display(cell_wide)

# Prospective localization decision

## 19. Total-effect replication and localization signals

First, v0.6.2 must independently retain the v0.6.1 **total expansion** direction:

- `CE - CC > 0`
- `FE - FC > 0`

with positive means in at least 3/4 language×length cells and leave-one-trial-out positive means.

Only if that total effect replicates do we interpret localization.

A component is called a **robust positive signal** only if, in both ID regimes:

- pooled mean > 0;
- 95% cluster-bootstrap CI lower bound > 0.

`LOCALIZATION` is then:

- `query_shift` if only query displacement is robust;
- `filler_spread` if only filler spreading is robust;
- `both` if both are robust;
- `mixed_or_unresolved` otherwise.

This is intentionally conservative.

In [ ]:
cc_rows = df[
    df['geometry_code']
    == 'CC'
]

canonical_acc = float(
    cc_rows['correct'].mean()
)

cc_pred_counts = (
    cc_rows['predicted_digit']
    .value_counts()
    .reindex(
        DIGITS,
        fill_value=0,
    )
)

canonical_max_pred_share = float(
    cc_pred_counts.max()
    / cc_pred_counts.sum()
)

pooled = pooled_summary.set_index(
    'effect'
)

def pooled_mean(effect):
    return float(
        pooled.loc[
            effect,
            'mean',
        ]
    )

def pooled_lo(effect):
    return float(
        pooled.loc[
            effect,
            'ci_lo',
        ]
    )

positive_cells = {}

for effect in EFFECT_NAMES:
    if effect in cell_wide.columns:
        positive_cells[effect] = int(
            (
                cell_wide[
                    effect
                ] > 0
            ).sum()
        )

loo_positive = {}

for effect in EFFECT_NAMES:
    loo_positive[effect] = bool(
        (
            loo[
                effect
            ] > 0
        ).all()
    )

TOTAL_EXPANSION_REPLICATION = bool(
    canonical_acc >= 0.90
    and canonical_max_pred_share <= 0.40
    and pooled_mean(
        'total_expansion_canonical'
    ) > 0
    and pooled_mean(
        'total_expansion_fragmented'
    ) > 0
    and positive_cells[
        'total_expansion_canonical'
    ] >= 3
    and positive_cells[
        'total_expansion_fragmented'
    ] >= 3
    and loo_positive[
        'total_expansion_canonical'
    ]
    and loo_positive[
        'total_expansion_fragmented'
    ]
)

QUERY_SHIFT_SIGNAL = bool(
    pooled_mean(
        'query_shift_canonical'
    ) > 0
    and pooled_mean(
        'query_shift_fragmented'
    ) > 0
    and pooled_lo(
        'query_shift_canonical'
    ) > 0
    and pooled_lo(
        'query_shift_fragmented'
    ) > 0
)

FILLER_SPREAD_SIGNAL = bool(
    pooled_mean(
        'filler_spread_canonical'
    ) > 0
    and pooled_mean(
        'filler_spread_fragmented'
    ) > 0
    and pooled_lo(
        'filler_spread_canonical'
    ) > 0
    and pooled_lo(
        'filler_spread_fragmented'
    ) > 0
)

if not TOTAL_EXPANSION_REPLICATION:
    LOCALIZATION = 'total_effect_not_replicated'
elif (
    QUERY_SHIFT_SIGNAL
    and FILLER_SPREAD_SIGNAL
):
    LOCALIZATION = 'both'
elif QUERY_SHIFT_SIGNAL:
    LOCALIZATION = 'query_shift'
elif FILLER_SPREAD_SIGNAL:
    LOCALIZATION = 'filler_spread'
else:
    LOCALIZATION = 'mixed_or_unresolved'

print('CC accuracy:', canonical_acc)
print(
    'CC prediction counts:',
    cc_pred_counts.to_dict(),
)
print(
    'CC max prediction share:',
    canonical_max_pred_share,
)

print('\nPooled component effects:')
for effect in [
    'query_shift_canonical',
    'query_shift_fragmented',
    'filler_spread_canonical',
    'filler_spread_fragmented',
    'total_expansion_canonical',
    'total_expansion_fragmented',
    'representation_source',
    'representation_query_shift',
    'representation_spread',
]:
    print(
        f"{effect:<34} "
        f"mean={pooled_mean(effect):+.4f} "
        f"CIlo={pooled_lo(effect):+.4f}"
    )

print(
    '\nPositive language×length cells:',
    {
        k: positive_cells[k]
        for k in [
            'query_shift_canonical',
            'query_shift_fragmented',
            'filler_spread_canonical',
            'filler_spread_fragmented',
            'total_expansion_canonical',
            'total_expansion_fragmented',
        ]
    },
)

print(
    'LOO positive:',
    {
        k: loo_positive[k]
        for k in [
            'query_shift_canonical',
            'query_shift_fragmented',
            'filler_spread_canonical',
            'filler_spread_fragmented',
            'total_expansion_canonical',
            'total_expansion_fragmented',
        ]
    },
)

print(
    '\nTOTAL_EXPANSION_REPLICATION =',
    TOTAL_EXPANSION_REPLICATION,
)
print(
    'QUERY_SHIFT_SIGNAL =',
    QUERY_SHIFT_SIGNAL,
)
print(
    'FILLER_SPREAD_SIGNAL =',
    FILLER_SPREAD_SIGNAL,
)
print(
    'LOCALIZATION =',
    LOCALIZATION,
)

if not TOTAL_EXPANSION_REPLICATION:
    print(
        '\nSTOP: the v0.6.1 total positional effect did not replicate again.'
    )
else:
    print(
        '\nTotal expansion effect replicated. '
        'Use LOCALIZATION to decide whether the mechanism is primarily '
        'query displacement, filler positional spreading, both, or unresolved.'
    )

## 20. Save audit

In [ ]:
pooled_effects = {}

for _, row in pooled_summary.iterrows():
    pooled_effects[
        row['effect']
    ] = {
        'mean': float(
            row['mean']
        ),
        'ci_lo': float(
            row['ci_lo']
        ),
        'ci_hi': float(
            row['ci_hi']
        ),
    }

audit = {
    'model': MODEL_NAME,
    'transformers_version': transformers.__version__,
    'notebook_version': 'v0.6.2',
    'task_version': TASK_VERSION,
    'scorer_version': SCORER_VERSION,
    'position_intervention': POSITION_INTERVENTION,
    'followup_status': 'prospective_new_seed_geometry_localization',
    'previous_v061_factorial_position_replication': (
        None
        if previous is None
        else previous.get(
            'factorial_position_replication'
        )
    ),
    'claim_scope': 'associative_retrieval',
    'frozen_n_records': N_RECORDS,
    'frozen_frag_keep_by_language': FRAG_KEEP_BY_LANGUAGE,
    'new_semantic_seeds': SEMANTIC_SEEDS,
    'target_lengths': TARGET_LENGTHS,
    'primary_metric': 'gold_vs_best_distractor_logit_margin',
    'gold_position_combinations_each_n': 2,
    'same_ids_CC_CQ_CE': True,
    'same_ids_FC_FQ_FE': True,
    'CQ_CE_same_query_readout_geometry': True,
    'FQ_FE_same_query_readout_geometry': True,
    'canonical_accuracy': canonical_acc,
    'canonical_max_pred_share': canonical_max_pred_share,
    'pooled_effects_cluster_bootstrap': pooled_effects,
    'positive_cells': positive_cells,
    'leave_one_trial_out_positive': loo_positive,
    'total_expansion_replication': bool(
        TOTAL_EXPANSION_REPLICATION
    ),
    'query_shift_signal': bool(
        QUERY_SHIFT_SIGNAL
    ),
    'filler_spread_signal': bool(
        FILLER_SPREAD_SIGNAL
    ),
    'localization': LOCALIZATION,
}

with AUDIT_PATH.open(
    'w',
    encoding='utf-8',
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    )
)

print('Saved:', AUDIT_PATH)

# Files to upload

From:

`MyDrive/qwen3_position_localization_v062/`

upload:

1. `v062_geometry_localization_4k8k.jsonl`
2. `v062_audit.json`

Key lines:

- `TOTAL_EXPANSION_REPLICATION = True/False`
- `QUERY_SHIFT_SIGNAL = True/False`
- `FILLER_SPREAD_SIGNAL = True/False`
- `LOCALIZATION = ...`

Do not start a second-model replication until this localization result is reviewed.